# MASTER: Market-Guided Stock Transformer for Stock Price Forecasting

## Introduction

This notebook is based on the research outlined in the paper "MASTER: Market-Guided Stock Transformer for Stock Price Forecasting" by Li, T., Liu, Z., Shen, Y., Wang, X., Chen, H., & Huang, S. (2024), published in the Proceedings of the AAAI Conference on Artificial Intelligence, 38(1), 162-170. The official implementation of the model can be found [here](https://github.com/SJTU-DMTai/MASTER).

### Model Application and Adaptations

For this dataset, the architecture of the MASTER model effectively condenses information across different time steps for the same stock symbol while considering the interactions between different symbols. This is achieved through multi-dimensional self-attention mechanisms aimed at predicting `responder_6`.

However, since the provided data does not include market features that would inform the filtering of other data, we have omitted the Gating Mechanism from the original model. In essence, this adaptation results in a simplified version referred to as (MA)STER: a stock transformer without the gating mechanism.

### Data Sampling

The implementation of the `TSDataSampler` class draws inspiration from Qlib library version 0.8.6, ensuring an appropriate sampling strategy for time series data.

### Focus on Data Characteristics

<font color=red>Taking into account the findings from our earlier Exploratory Data Analysis (EDA), I will tailor the application of the model according to the characteristics of our dataset. This notebook will focus extensively on data preprocessing and analysis, as well as the practical application of the model. Our aim is to leverage the specific features of the data to optimize model performance.


For a deeper understanding of the MASTER model's architecture and functionality, please refer to the [original paper](https://doi.org/10.1609/aaai.v38i1.27767) and the [official code repository](https://github.com/SJTU-DMTai/MASTER).


## Import

In [3]:
# Author of this notebook: Xinlei Hao
import copy
import torch
import torch.optim as optim

import numpy as np
import pandas as pd
import polars as pl

from typing import Tuple, Union, List
from copy import deepcopy
from tqdm import tqdm
import bisect
from torch.utils.data import DataLoader
from torch.utils.data import Sampler
from torch import nn
from torch.nn.modules.linear import Linear
from torch.nn.modules.dropout import Dropout
from torch.nn.modules.normalization import LayerNorm
import math

import warnings
warnings.filterwarnings("ignore")

## Utils

In [4]:
def lazy_sort_index(df: pd.DataFrame, axis=0) -> pd.DataFrame:
    idx = df.index if axis == 0 else df.columns
    if (
        not idx.is_monotonic_increasing
        and isinstance(idx, pd.MultiIndex)
        and not idx.is_lexsorted()
    ):  
        return df.sort_index(axis=axis)
    else:
        return df

def np_ffill(arr: np.array):
    mask = np.isnan(arr.astype(float))  # np.isnan only works on np.float
    # get fill index
    idx = np.where(~mask, np.arange(mask.shape[0]), 0)
    np.maximum.accumulate(idx, out=idx)
    return arr[idx]

## Data Sampling and Dataset Preparation

As mentioned earlier, the `TSDataSampler` class and `DailyBatchSamplerRandom` class is implemented by drawing inspiration from [Qlib](https://github.com/microsoft/qlib). The `TSDataSampler` class is designed to construct a dataset that includes the `time_step` dimension, which is essential for feeding time-series data into the model for training and prediction purposes. 




### TSDataSampler (Dataset)

The `TSDataSampler` facilitates the creation of a dataset that not only captures the temporal dynamics of stock prices but also organizes the data in a way that aligns with the requirements of the MASTER model. By incorporating the `time_step` dimension, it ensures that each data point retains its sequential context, allowing the model to effectively learn patterns over time.
 
Key Features:
- Time Step Sampling: Allows sampling of data with a specified time step.
- Fill Missing Values: Provides options to fill missing values using forward fill and backward fill methods. **In our dataset, there is no need to worry about filling missing values because this task has already been completed during the data input process. The methods and procedures for handling missing values are detailed in `notebooks/EDA_Part3_fillna.ipynb`.**

Usage:
- The TSDataSampler class can be used to create datasets that are compatible with time-series models.

In [5]:
# v0.8.6
class TSDataSampler:
    """
    (T)ime-(S)eries DataSampler
    This is the processed_data of TSDatasetH

    It works like `torch.data.utils.Dataset`, it provides a very convenient interface for constructing time-series
    dataset based on tabular data.
    - On time step dimension, the smaller index indicates the historical data and the larger index indicates the future
      data.

    If user have further requirements for processing data, user could process them based on `TSDataSampler` or create
    more powerful subclasses.

    Known Issues:
    - For performance issues, this Sampler will convert dataframe into arrays for better performance. This could processed_data
      in a different data type

    """

    def __init__(
            self, data: pd.DataFrame, start, end, step_len: int, fillna_type: str = "none", dtype=None, flt_data=None
    ):
        """
        Build a dataset which looks like torch.data.utils.Dataset.

        Parameters
        ----------
        data : pd.DataFrame
            The raw tabular data
        start :
            The indexable start time
        end :
            The indexable end time
        step_len : int
            The length of the time-series step
        fillna_type : int
            How will qlib handle the sample if there is on sample in a specific date.
            none:
                fill with np.nan
            ffill:
                ffill with previous sample
            ffill+bfill:
                ffill with previous samples first and fill with later samples second
        flt_data : pd.Series
            a column of data(True or False) to filter data.
            None:
                kepp all data

        """
        self.start = start
        self.end = end
        self.step_len = step_len
        self.fillna_type = fillna_type
        #assert get_level_index(data, "datetime") == 0
        self.data = lazy_sort_index(data)

        kwargs = {"object": self.data}
        if dtype is not None:
            kwargs["dtype"] = dtype

        self.data_arr = np.array(**kwargs)  # Get index from numpy.array will much faster than DataFrame.values!
        # NOTE:
        # - append last line with full NaN for better performance in `__getitem__`
        # - Keep the same dtype will processed_data in a better performance
        self.data_arr = np.append(
            self.data_arr, np.full((1, self.data_arr.shape[1]), np.nan, dtype=self.data_arr.dtype), axis=0
        )
        self.nan_idx = -1  # The last line is all NaN

        # the data type will be changed
        # The index of usable data is between start_idx and end_idx
        self.idx_df, self.idx_map = self.build_index(self.data)
        self.data_index = deepcopy(self.data.index)

        if flt_data is not None:
            if isinstance(flt_data, pd.DataFrame):
                assert len(flt_data.columns) == 1
                flt_data = flt_data.iloc[:, 0]
            # NOTE: bool(np.nan) is True !!!!!!!!
            # make sure reindex comes first. Otherwise extra NaN may appear.
            flt_data = flt_data.reindex(self.data_index).fillna(False).astype(np.bool)
            self.flt_data = flt_data.values
            self.idx_map = self.flt_idx_map(self.flt_data, self.idx_map)
            self.data_index = self.data_index[np.where(self.flt_data)[0]]
        self.idx_map = self.idx_map2arr(self.idx_map)

        # self.start_idx, self.end_idx = self.data_index.slice_locs(
        #     start=time_to_slc_point(start), end=time_to_slc_point(end)
        # )
        #self.start_idx, self.end_idx = 0, len(self.idx_map) # 这里进行了非常粗暴的改动
        # Find the index positions of start and end in the level 0 index
        # 这里下面两行进行了改动
        self.start_idx = self.data_index.get_level_values(0).searchsorted(start, side='left')
        self.end_idx = self.data_index.get_level_values(0).searchsorted(end, side='right')
        
        self.idx_arr = np.array(self.idx_df.values, dtype=np.float64)  # for better performance

        del self.data  # save memory

    @staticmethod
    def idx_map2arr(idx_map):
        # pytorch data sampler will have better memory control without large dict or list
        # - https://github.com/pytorch/pytorch/issues/13243
        # - https://github.com/airctic/icevision/issues/613
        # So we convert the dict into int array.
        # The arr_map is expected to behave the same as idx_map

        dtype = np.int32
        # set a index out of bound to indicate the none existing
        no_existing_idx = (np.iinfo(dtype).max, np.iinfo(dtype).max)

        max_idx = max(idx_map.keys())
        arr_map = []
        for i in range(max_idx + 1):
            arr_map.append(idx_map.get(i, no_existing_idx))
        arr_map = np.array(arr_map, dtype=dtype)
        return arr_map

    @staticmethod
    def flt_idx_map(flt_data, idx_map):
        idx = 0
        new_idx_map = {}
        for i, exist in enumerate(flt_data):
            if exist:
                new_idx_map[idx] = idx_map[i]
                idx += 1
        return new_idx_map

    def get_index(self):
        """
        Get the pandas index of the data, it will be useful in following scenarios
        - Special sampler will be used (e.g. user want to sample day by day)
        """
        return self.data_index[self.start_idx: self.end_idx]

    def config(self, **kwargs):
        # Config the attributes
        for k, v in kwargs.items():
            setattr(self, k, v)

    @staticmethod
    def build_index(data: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
        """
        The relation of the data

        Parameters
        ----------
        data : pd.DataFrame
            The dataframe with <datetime, DataFrame>

        Returns
        -------
        Tuple[pd.DataFrame, dict]:
            1) the first element:  reshape the original index into a <datetime(row), instrument(column)> 2D dataframe
                instrument SH600000 SH600004 SH600006 SH600007 SH600008 SH600009  ...
                datetime
                2021-01-11        0        1        2        3        4        5  ...
                2021-01-12     4146     4147     4148     4149     4150     4151  ...
                2021-01-13     8293     8294     8295     8296     8297     8298  ...
                2021-01-14    12441    12442    12443    12444    12445    12446  ...
            2) the second element:  {<original index>: <row, col>}
        """
        # object incase of pandas converting int to float
        idx_df = pd.Series(range(data.shape[0]), index=data.index, dtype=object)
        idx_df = lazy_sort_index(idx_df.unstack())
        # NOTE: the correctness of `__getitem__` depends on columns sorted here
        idx_df = lazy_sort_index(idx_df, axis=1)

        idx_map = {}
        for i, (_, row) in enumerate(idx_df.iterrows()):
            for j, real_idx in enumerate(row):
                if not np.isnan(real_idx):
                    idx_map[real_idx] = (i, j)
        return idx_df, idx_map

    @property
    def empty(self):
        return len(self) == 0

    def _get_indices(self, row: int, col: int) -> np.array:
        """
        get series indices of self.data_arr from the row, col indices of self.idx_df

        Parameters
        ----------
        row : int
            the row in self.idx_df
        col : int
            the col in self.idx_df

        Returns
        -------
        np.array:
            The indices of data of the data
        """
        indices = self.idx_arr[max(row - self.step_len + 1, 0): row + 1, col]

        if len(indices) < self.step_len:
            indices = np.concatenate([np.full((self.step_len - len(indices),), np.nan), indices])

        if self.fillna_type == "ffill":
            indices = np_ffill(indices)
        elif self.fillna_type == "ffill+bfill":
            indices = np_ffill(np_ffill(indices)[::-1])[::-1]
        else:
            assert self.fillna_type == "none"
        return indices

    def _get_row_col(self, idx) -> Tuple[int]:
        """
        get the col index and row index of a given sample index in self.idx_df

        Parameters
        ----------
        idx :
            the input of  `__getitem__`

        Returns
        -------
        Tuple[int]:
            the row and col index
        """
        # The the right row number `i` and col number `j` in idx_df
        if isinstance(idx, (int, np.integer)):
            real_idx = self.start_idx + idx
            if self.start_idx <= real_idx < self.end_idx:
                i, j = self.idx_map[real_idx]  # TODO: The performance of this line is not good
            else:
                raise KeyError(f"{real_idx} is out of [{self.start_idx}, {self.end_idx})")
        elif isinstance(idx, tuple):
            # <TSDataSampler object>["datetime", "instruments"]
            date, inst = idx
            date = pd.Timestamp(date)
            i = bisect.bisect_right(self.idx_df.index, date) - 1
            # NOTE: This relies on the idx_df columns sorted in `__init__`
            j = bisect.bisect_left(self.idx_df.columns, inst)
        else:
            raise NotImplementedError(f"This type of input is not supported")
        return i, j



    def __getitem__(self, idx: Union[int, Tuple[object, str], List[int]]):
        """
        # We have two method to get the time-series of a sample
        tsds is a instance of TSDataSampler

        # 1) sample by int index directly
        tsds[len(tsds) - 1]

        # 2) sample by <datetime,instrument> index
        tsds['2016-12-31', "SZ300315"]

        # The return value will be similar to the data retrieved by following code
        df.loc(axis=0)['2015-01-01':'2016-12-31', "SZ300315"].iloc[-30:]

        Parameters
        ----------
        idx : Union[int, Tuple[object, str]]
        """
        # Multi-index type
        mtit = (list, np.ndarray)
        if isinstance(idx, mtit):
            indices = [self._get_indices(*self._get_row_col(i)) for i in idx]
            indices = np.concatenate(indices)
        else:
            indices = self._get_indices(*self._get_row_col(idx))

        indices = np.nan_to_num(indices.astype(np.float64), nan=self.nan_idx).astype(int)

        data = self.data_arr[indices]
        if isinstance(idx, mtit):
            # if we get multiple indexes, addition dimension should be added.
            # <sample_idx, step_idx, feature_idx>
            data = data.reshape(-1, self.step_len, *data.shape[1:])
        return data
    
    def __len__(self):
        return len(self.idx_map)

### DailyBatchSamplerRandom
·DailyBatchSamplerRandom· is a data sampler class designed for time-series data in [Qlib](https://github.com/microsoft/qlib). It randomly samples daily batches (**in our data is batch for every time_id**) from the dataset, ensuring that each batch contains data from a single day. This class is useful for training models on time-series data where the temporal order within each day is important, but the order of days can be randomized. 
 
Key Features:
- Random Sampling: Randomly selects daily batches from the dataset.
- Batch Consistency: Ensures that each batch contains data from a single day(**in our data is each time_id**).
- Time-Series Compatibility: Designed specifically for time-series data, maintaining the temporal order within each day.

Usage:
- This class can be used with PyTorch's DataLoader to create data loaders that provide daily batches of time-series data for training and evaluation.

In [6]:
class DailyBatchSamplerRandom(Sampler):
    def __init__(self, data_source, shuffle=False):
        self.data_source = data_source
        self.shuffle = shuffle
        # calculate number of samples in each batch
        self.daily_count = pd.Series(index=self.data_source.get_index(), dtype=pd.Float32Dtype()).groupby("time_id").size().values
        self.daily_index = np.roll(np.cumsum(self.daily_count), 1)  # calculate begin index of each batch
        self.daily_index[0] = 0

    def __iter__(self):
        if self.shuffle:
            index = np.arange(len(self.daily_count))
            np.random.shuffle(index)
            for i in index:
                yield np.arange(self.daily_index[i], self.daily_index[i] + self.daily_count[i])
        else:
            for idx, count in zip(self.daily_index, self.daily_count):
                yield np.arange(idx, idx + count)

    def __len__(self):
        return len(self.data_source)

## Data preperation \& DataLoader

#### Incorporating Insights and Preprocessed Data

This section will provide a detailed guide on how to load the model into the dataloader, leveraging the findings from `notebooks/EDA_Part1.ipynb` and `notebooks/EDA_Part2.ipynb`, as well as the preprocessed data handled in `notebooks/EDA_Part3_fillna.ipynb`. Additionally, it will incorporate the utilities mentioned above to ensure a seamless integration process.

#### Overview

Building upon the exploratory data analysis (EDA) conducted in the initial notebooks, we have gained valuable insights into the dataset's structure and characteristics. These insights inform our approach to preprocessing and handling the data, ensuring that it is optimally prepared for model training and evaluation.

The `EDA_Part3_fillna.ipynb` notebook specifically addresses the issue of missing values, ensuring that all necessary data imputation has been performed. This step is critical for maintaining the integrity and reliability of the dataset, which in turn enhances the performance of the model.



In [7]:
data = pl.read_parquet('../processed_data/Processed_data_1695.parquet').to_pandas()
data

,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,...,feature_78,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,...,0.069660,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,...,-0.373929,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,...,0.109544,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,...,-0.146268,0.047888,-0.014610,-0.334644,-0.042600,-0.569183,-0.244618,-0.055909,-0.570931,-0.120065
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,...,-0.228050,-0.225697,-0.016521,-0.308620,0.205822,0.412048,-0.213416,0.532245,0.483285,-0.173360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150035,1698,967,34,3.242493,2.525160,-0.721981,2.544025,2.477615,0.417557,0.785812,...,0.016936,0.243475,0.166927,0.384940,-0.174297,-0.066046,-0.038767,-0.132337,-0.022426,-0.252461
150036,1698,967,35,1.079139,1.857906,-0.790646,2.745439,2.339877,0.845065,0.651370,...,0.050860,0.850152,0.909382,1.015314,0.235962,0.122539,0.099559,-0.249584,-0.123571,-0.460630
150037,1698,967,36,1.033172,2.515527,-0.672298,2.289250,2.521592,0.255077,0.919892,...,0.152333,0.395684,-0.292574,-3.215846,-0.535129,-0.178484,-1.808150,-0.065355,-0.000367,-0.125170
150038,1698,967,37,1.243116,2.663298,-0.889112,2.313155,3.101428,0.324454,0.618944,...,-0.029483,1.925987,0.479394,3.621867,-0.107114,-0.063599,1.204755,-0.148711,-0.026583,-0.256395


In [8]:
data = data.drop([f'responder_{i}' for i in [0,1,2,3,4,5,7,8]], axis=1)
data 

,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,...,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_6
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,...,-0.841593,0.311180,-0.575253,0.171726,0.140167,1.225793,0.929635,0.067034,0.069660,0.167316
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,...,-0.760508,0.142326,-0.745543,0.171726,0.140167,-0.226083,-0.213198,-0.206452,-0.373929,0.730008
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,...,-0.888185,1.507160,-0.264173,0.171726,0.140167,1.517169,1.503251,0.094979,0.109544,-1.758799
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,...,-1.066270,0.206620,-0.518951,0.171726,0.140167,0.425632,0.424689,-0.129259,-0.146268,-0.055909
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,...,-1.215103,0.660906,-0.331937,0.171726,0.140167,-0.064459,-0.087876,-0.265680,-0.228050,0.532245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150035,1698,967,34,3.242493,2.525160,-0.721981,2.544025,2.477615,0.417557,0.785812,...,1.333250,1.075499,1.798264,-0.183443,-0.190222,0.234211,0.347142,-0.044463,0.016936,-0.132337
150036,1698,967,35,1.079139,1.857906,-0.790646,2.745439,2.339877,0.845065,0.651370,...,-0.180839,-0.086100,-0.153405,-0.196077,-0.175292,1.045780,0.739733,0.033720,0.050860,-0.249584
150037,1698,967,36,1.033172,2.515527,-0.672298,2.289250,2.521592,0.255077,0.919892,...,0.860160,0.024223,0.374852,-0.220933,-0.161584,0.032771,0.036888,0.168908,0.152333,-0.065355
150038,1698,967,37,1.243116,2.663298,-0.889112,2.313155,3.101428,0.324454,0.618944,...,0.478357,0.782692,0.581421,-0.106056,-0.111017,0.163867,0.169331,-0.037563,-0.029483,-0.148711


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150040 entries, 0 to 150039
Data columns (total 84 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   date_id      150040 non-null  int16  
 1   time_id      150040 non-null  int16  
 2   symbol_id    150040 non-null  int8   
 3   weight       150040 non-null  float32
 4   feature_00   150040 non-null  float32
 5   feature_01   150040 non-null  float32
 6   feature_02   150040 non-null  float32
 7   feature_03   150040 non-null  float32
 8   feature_04   150040 non-null  float32
 9   feature_05   150040 non-null  float32
 10  feature_06   150040 non-null  float32
 11  feature_07   150040 non-null  float32
 12  feature_08   150040 non-null  float32
 13  feature_09   150040 non-null  float64
 14  feature_10   150040 non-null  float64
 15  feature_11   150040 non-null  float64
 16  feature_12   150040 non-null  float32
 17  feature_13   150040 non-null  float32
 18  feature_14   150040 non-

### Important Issue 1: Changes in Trading Symbols Across Days

#### Problem Statement

The MASTER model utilizes the Transformer to capture the attention between different stocks (referred to as `symbol_id` in this dataset). When using the `DailyBatchSamplerRandom` class, one dimension corresponds to `symbol_id`. According to our Exploratory Data Analysis (EDA), it was observed that `symbol_id` changes across days; the set of symbols traded varies from day to day. This means that the dimensionality of `symbol_id` for each sample fluctuates—some days have 36 symbols, others have 38. Consequently, this variability complicates batch splitting. Setting `batch_size=1`, as in the original MASTER model, where an entire day's time-stepped panel data is treated as a single batch, can significantly slow down training.

#### Solution Approach

To address this issue, I drew inspiration from two observations:

1. **Symbol ID Range**: During EDA, it was noted that `symbol_id` ranges from 0 to 39, indicating that there are 40 unique symbols traded over the 1699-day period.
2. **Padding in Image Processing**: In deep learning applications for image processing, padding is commonly used to add pixels at the edges of images, maintaining consistent dimensions.

Additionally, the target variable `responder_6` falls within the range [-5, 5], which likely represents returns or yield.

#### Implementation of Padding

Inspired by these insights, I decided to apply padding to the dataset to fix the length of the `symbol_id` dimension to 40. For days when a particular symbol does not trade, all its entries are padded with zeros (`responder_6 = 0` implies no return, which aligns with the fact that the product did not trade that day). This approach ensures consistency in the `symbol_id` dimension, allowing us to use the `DailyBatchSamplerRandom` class while freely setting the `batch_size` parameter. We can now select any number of `time_id` panel data points to update model parameters, thereby enhancing the model's flexibility and efficiency.

By adopting this strategy, we overcome the challenge of varying symbol dimensions across days and improve the training process without sacrificing model performance.


In [10]:
def pad_data(data, symbol_range=range(40)):
    # Get all unique date_ids and time_ids
    date_ids = data['date_id'].unique()
    time_ids = data['time_id'].unique()

    # Create a MultiIndex with all combinations of date_id, time_id, and symbol_id
    full_index = pd.MultiIndex.from_product([date_ids, time_ids, symbol_range], names=['date_id', 'time_id', 'symbol_id'])

    # Set the index to ['date_id', 'time_id', 'symbol_id']
    data = data.set_index(['date_id', 'time_id', 'symbol_id'])

    # Reindex the data to include all combinations, filling missing values with 0
    padded_data = data.reindex(full_index, fill_value=0)

    return padded_data

# Apply padding
data = pad_data(data)

# Reset index if needed
data = data.reset_index()

In [11]:
data.shape

(154880, 84)

In [12]:
data

,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,...,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_6
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,...,-0.841593,0.311180,-0.575253,0.171726,0.140167,1.225793,0.929635,0.067034,0.069660,0.167316
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,...,-0.760508,0.142326,-0.745543,0.171726,0.140167,-0.226083,-0.213198,-0.206452,-0.373929,0.730008
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,...,-0.888185,1.507160,-0.264173,0.171726,0.140167,1.517169,1.503251,0.094979,0.109544,-1.758799
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,...,-1.066270,0.206620,-0.518951,0.171726,0.140167,0.425632,0.424689,-0.129259,-0.146268,-0.055909
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,...,-1.215103,0.660906,-0.331937,0.171726,0.140167,-0.064459,-0.087876,-0.265680,-0.228050,0.532245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154875,1698,967,35,1.079139,1.857906,-0.790646,2.745439,2.339877,0.845065,0.651370,...,-0.180839,-0.086100,-0.153405,-0.196077,-0.175292,1.045780,0.739733,0.033720,0.050860,-0.249584
154876,1698,967,36,1.033172,2.515527,-0.672298,2.289250,2.521592,0.255077,0.919892,...,0.860160,0.024223,0.374852,-0.220933,-0.161584,0.032771,0.036888,0.168908,0.152333,-0.065355
154877,1698,967,37,1.243116,2.663298,-0.889112,2.313155,3.101428,0.324454,0.618944,...,0.478357,0.782692,0.581421,-0.106056,-0.111017,0.163867,0.169331,-0.037563,-0.029483,-0.148711
154878,1698,967,38,3.193685,2.728506,-0.745238,2.788789,2.343393,0.454731,0.862839,...,0.462717,0.799635,0.706102,-0.376377,-0.286764,-0.359046,-0.246135,-0.288941,-0.247774,-0.138548


Here, the observation is more than before.

### Important Issue 2: Encoding of Indices

#### Problem Statement

Unlike the original model's dataset, our dataset has three-dimensional indices: `date_id`, `time_id`, and `symbol_id`. Since both `date_id` and `time_id` are time-based indices, they can be merged. The challenge is how to merge them in such a way that the distinction between `date_id` and `time_id` is preserved, while still allowing for effective data splitting based on this combined index.

#### Solution Approach

Based on the findings from Exploratory Data Analysis (EDA), it was observed that the `time_id` for each trading day does not exceed 1000 over the whole period. Therefore, we propose encoding `date_id` and `time_id` into a single numerical index where:

- **Lower Digits (Units, Tens, Hundreds)**: Store the `time_id`.
- **Higher Digits**: Store the `date_id`.

The combined index format will be `date_idtime_id.0`, ensuring that the fractional part retains the original `time_id` value. For example, `1300900.0` represents `date_id = 1300` and `time_id = 900`.


- **Combination Logic**: 
  - Concatenate `date_id` and `time_id` into a single floating-point number.
  - The integer part will consist of `date_id` followed by `time_id`.
  - The fractional part will always be `.0` to preserve the original `time_id` value.

- **Example**:
  - If `date_id = 1300` and `time_id = 900`, the combined index becomes `1300900.0`.

This approach ensures that:
- The `date_id` and `time_id` remain distinguishable within the same index.
- The temporal characteristics of both indices are preserved.
- Data can be efficiently split and processed using this unified index.




In [13]:
data['time_id'] =((data['date_id'].astype(np.float64))*1000+(data['time_id'].astype(np.float64))).astype(np.float32)
data = data.drop(['date_id'], axis=1)
# to float32
data = data.astype(np.float32)
# set multi-index, first level is date_time_id, second level is stock_id
data = data.set_index(['time_id', 'symbol_id'])
data

weight  feature_00  feature_01  feature_02  feature_03  \
time_id   symbol_id                                                             
1695000.0 0.0        3.373552    2.776059    1.035769    1.790385    1.915069   
          1.0        2.802384    1.901147    1.029203    2.146977    2.811388   
          2.0        2.506616    2.801315    1.376992    2.313733    2.135918   
          3.0        1.868808    2.492323    1.293729    2.051276    2.601039   
          4.0        2.826087    2.754668    0.981692    2.405406    2.504181   
...                       ...         ...         ...         ...         ...   
1698967.0 35.0       1.079139    1.857906   -0.790646    2.745439    2.339877   
          36.0       1.033172    2.515527   -0.672298    2.289250    2.521592   
          37.0       1.243116    2.663298   -0.889112    2.313155    3.101428   
          38.0       3.193685    2.728506   -0.745238    2.788789    2.343393   
          39.0       0.000000    0.000000    0.000000    0.000000    0.000000   

                     feature_04  feature_05  feature_06  feature_07  \
time_id   symbol_id                                                   
1695000.0 0.0          1.765592   -0.105391   -0.169630   -0.345125   
          1.0          1.423654   -0.105936   -0.173366   -0.382062   
          2.0          1.765473   -0.173767   -0.261385   -0.473618   
          3.0          2.115296   -0.137015   -0.154580   -0.398430   
          4.0          2.060546   -0.067486   -0.153702   -0.205151   
...                         ...         ...         ...         ...   
1698967.0 35.0         0.845065    0.651370    1.180301    1.966379   
          36.0         0.255077    0.919892    1.172018    2.180496   
          37.0         0.324454    0.618944    1.185663    1.599724   
          38.0         0.454731    0.862839    0.964795    2.089673   
          39.0         0.000000    0.000000    0.000000    0.000000   

                     feature_08  ...  feature_70  feature_71  feature_72  \
time_id   symbol_id              ...                                       
1695000.0 0.0          0.037630  ...   -0.841593    0.311180   -0.575253   
          1.0          0.045153  ...   -0.760508    0.142326   -0.745543   
          2.0          0.018804  ...   -0.888185    1.507160   -0.264173   
          3.0          0.058264  ...   -1.066270    0.206620   -0.518951   
          4.0          0.021181  ...   -1.215103    0.660906   -0.331937   
...                         ...  ...         ...         ...         ...   
1698967.0 35.0         0.321543  ...   -0.180839   -0.086100   -0.153405   
          36.0         0.248460  ...    0.860160    0.024223    0.374852   
          37.0         0.319719  ...    0.478357    0.782692    0.581421   
          38.0         0.344931  ...    0.462717    0.799635    0.706102   
          39.0         0.000000  ...    0.000000    0.000000    0.000000   

                     feature_73  feature_74  feature_75  feature_76  \
time_id   symbol_id                                                   
1695000.0 0.0          0.171726    0.140167    1.225793    0.929635   
          1.0          0.171726    0.140167   -0.226083   -0.213198   
          2.0          0.171726    0.140167    1.517169    1.503251   
          3.0          0.171726    0.140167    0.425632    0.424689   
          4.0          0.171726    0.140167   -0.064459   -0.087876   
...                         ...         ...         ...         ...   
1698967.0 35.0        -0.196077   -0.175292    1.045780    0.739733   
          36.0        -0.220933   -0.161584    0.032771    0.036888   
          37.0        -0.106056   -0.111017    0.163867    0.169331   
          38.0        -0.376377   -0.286764   -0.359046   -0.246135   
          39.0         0.000000    0.000000    0.000000    0.000000   

                     feature_77  feature_78  responder_6  
time_id   symbol_id                                       
1695000.0 0.0 

In [14]:
data.info() # ensure that the data type is float32

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 154880 entries, (1695000.0, 0.0) to (1698967.0, 39.0)
Data columns (total 81 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   weight       154880 non-null  float32
 1   feature_00   154880 non-null  float32
 2   feature_01   154880 non-null  float32
 3   feature_02   154880 non-null  float32
 4   feature_03   154880 non-null  float32
 5   feature_04   154880 non-null  float32
 6   feature_05   154880 non-null  float32
 7   feature_06   154880 non-null  float32
 8   feature_07   154880 non-null  float32
 9   feature_08   154880 non-null  float32
 10  feature_09   154880 non-null  float32
 11  feature_10   154880 non-null  float32
 12  feature_11   154880 non-null  float32
 13  feature_12   154880 non-null  float32
 14  feature_13   154880 non-null  float32
 15  feature_14   154880 non-null  float32
 16  feature_15   154880 non-null  float32
 17  feature_16   154880 non-null  float32
 18

In [15]:
# Through the above processing, the data has been converted to the required format, and we can search the index for the corresponding time_id in the way like this:
data.index.get_level_values(0).searchsorted(1698967.0, side='left') 

154840

In [16]:
max_time_id = data.index.get_level_values('time_id').max() # get the maximum time_id
min_time_id = data.index.get_level_values('time_id').min() # get the minimum time_id

print("max_time_id", max_time_id)
print("min_time_id", min_time_id)

# split 4:1, first take int and then convert to the same data format as max_time_id
split_time_id = type(max_time_id)(min_time_id+int((max_time_id - min_time_id) * 0.8))
split_time_id2 = type(max_time_id)(min_time_id+int((max_time_id - min_time_id) * 0.9))
print("split_time_id", split_time_id)
print("split_time_id2", split_time_id2)

train_data = data.query('time_id <= @split_time_id').copy()
valid_data = data.query('time_id > @split_time_id & time_id <= @split_time_id2').copy()
test_data = data.query('time_id > @split_time_id2').copy()

max_time_id 1698967.0
min_time_id 1695000.0
split_time_id 1698173.0
split_time_id2 1698570.0


In [17]:
type(split_time_id)

numpy.float32

#### Final step for data prepereation
Having completed all the necessary preparations, including data preprocessing, handling missing values, encoding indices, and addressing changes in trading symbols across days, we are now ready to package the data into a `Dataset` and integrate it with a `Dataloader`.

In [18]:
train_dataset = TSDataSampler(data=train_data, start=min_time_id, end=split_time_id, step_len=10, fillna_type='ffill+bfill',)
valid_dataset = TSDataSampler(data=valid_data, start=split_time_id+1, end=split_time_id2, step_len=10, fillna_type='ffill+bfill',)
test_dataset = TSDataSampler(data=test_data, start=split_time_id2+1, end=max_time_id, step_len=10,
                              fillna_type='ffill+bfill', )

train_sampler = DailyBatchSamplerRandom(train_dataset, shuffle=False)
valid_sampler = DailyBatchSamplerRandom(valid_dataset, shuffle=False)
test_sampler = DailyBatchSamplerRandom(test_dataset, shuffle=False)

train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=50)
valid_loader = DataLoader(valid_dataset, sampler=valid_sampler, batch_size=50)
test_loader = DataLoader(test_dataset, sampler=test_sampler, batch_size=50)

In [19]:
# Iterate over the first ten batches in the train_loader and print their shapes
for i, data in enumerate(train_loader):
    if i >= 10:
        break
    print(data.shape)

torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])
torch.Size([50, 40, 10, 81])


# Model

The following describes the customized modifications performed on the MASTER model within the official codebase from [this GitHub repository](https://github.com/SJTU-DMTai/MASTER). The changes were implemented with the aim of tailoring the model to better fit specific requirements and improve its performance. Specifically, the customization involved:

- **Eliminating the Gate Mechanism:** The original architecture featured a gate mechanism designed to control information flow. This component has been removed to simplify the model structure and potentially reduce computational overhead.

- **Implementing Early Stopping:** An early stopping strategy has been introduced to halt the training process when the model's performance on a validation set no longer improves after a certain number of epochs. This prevents overfitting and saves computational resources.

- **Adapting the Loss Function:** The loss function has been revised to utilize a sample-weighted zero-mean R-squared score ($R^2$), specifically for the \texttt{responder_6} predictions. The calculation for this metric is given by:
  
  $$
  R^2 = 1 - \frac{\sum w_i (y_i - \hat{y}_i)^2}{\sum w_i y_i^2}
  $$

  Here, $y_i$ represents the true values, $\hat{y}_i$ are the predicted values for \texttt{responder_6}, and $w_i$ indicates the weight associated with each sample. By incorporating these weights into the loss function, the model can be made more sensitive to certain types of errors or data points, thereby improving prediction accuracy on critical instances.

Furthermore, to aid in understanding the underlying mechanics of the modified model, comprehensive annotations have been integrated directly into the code, providing clarity on the purpose and functionality of various components.


## SequenceModel (Base model)

In [20]:
def calc_weighted_r2(pred, label, weight):
    """
    Calculate the weighted R-squared (R2) score.

    Parameters:
    pred (array-like): Predicted values.
    label (array-like): True values.
    weight (array-like): Weights for each sample.

    Returns:
    float: Weighted R-squared score.
    """
    df = pd.DataFrame({'pred': pred, 'label': label, 'weight': weight})
    mask = ~df['label'].isna()
    df = df[mask]
    
    numerator = np.sum(df['weight'] * (df['label'] - df['pred']) ** 2)
    denominator = np.sum(df['weight'] * df['label'] ** 2)
    weighted_r2 = 1 - numerator / denominator
    return weighted_r2

In [21]:
class SequenceModel():
    def __init__(self, n_epochs, lr, GPU=None, seed=None, train_stop_loss_thred=None, save_path='../models/', save_prefix=''):
        self.n_epochs = n_epochs
        self.lr = lr
        self.device = torch.device(
            'mps' if torch.backends.mps.is_available() else f"cuda:{GPU}" if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        self.seed = seed
        self.train_stop_loss_thred = train_stop_loss_thred

        if self.seed is not None:
            np.random.seed(self.seed)
            torch.manual_seed(self.seed)
        self.fitted = False
        self.model = None
        self.train_optimizer = None

        self.save_path = save_path
        self.save_prefix = save_prefix

    def init_model(self):
        if self.model is None:
            raise ValueError("models has not been initialized")

        self.train_optimizer = optim.Adam(self.model.parameters(), self.lr)
        self.model.to(self.device)

    def loss_fn(self, pred, label, weights):
        '''
        '''
        mask = ~torch.isnan(label)
        pred = pred[mask]
        label = label[mask]
        weights = weights[mask]

        numerator = torch.sum(weights * (label - pred) ** 2)
        denominator = torch.sum(weights * label ** 2)
        r2 = 1 - numerator / denominator

        return -r2  # Since we want to minimize the loss, return the negative R-squared value

    def train_epoch(self, data_loader):
        '''
        This method is used to train the deep learning models for one complete epoch.
        :param data_loader: DataLoader providing batches of data.
        :return: Average loss for the epoch.
        '''
        self.model.train()
        losses = []  # Initialize an empty list to store the loss for each batch.
        for data in tqdm(data_loader, desc="Training", leave=False):
            # Iterate over batches provided by the DataLoader.
            data = torch.squeeze(data, dim=0)
            '''
            data.shape: (N, T, F)
            N - number of stocks
            T - length of lookback_window, 10, data is processed into a specific shape tensor usually in the __getitem__ method of the Dataset class
            F - length of feature, here is 80 (including weights)
            '''
            # The feature and label variables store the input features and target labels, respectively, extracted from the adjusted data tensor and moved to the specified device (e.g., GPU).
            feature = data[:, :, 1:-1].to(self.device)
            label = data[:, -1, -1].to(self.device)
            weights = data[:, -1, 0].to(self.device)  # Assuming weights are at index 0 in the 4th dimension

            pred = self.model(feature.float())  # Model prediction: Convert the data type of the feature tensor to float. This is because PyTorch models usually require float inputs for computation. If the feature is not of float type (e.g., integer or long), it needs to be converted to ensure the models can correctly process the input data.
            loss = self.loss_fn(pred, label, weights)
            losses.append(loss.item())

            # Zero the gradients, perform backpropagation, and update the models parameters.
            self.train_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_value_(self.model.parameters(), 3.0)  # Clip gradients to prevent exploding gradients. This is the clipping threshold. All parameter gradient values will be limited between -3.0 and 3.0. It is used to clip gradients to a specified maximum or minimum value. This prevents gradients from becoming too large during backpropagation, which can lead to numerical instability. Generally set between 1.0-5.0.
            self.train_optimizer.step()

        return float(np.mean(losses))

    def test_epoch(self, data_loader):
        self.model.eval()
        losses = []

        for data in tqdm(data_loader, desc="Validation", leave=False):
            data = torch.squeeze(data, dim=0)
            feature = data[:, :, 1:-1].to(self.device)
            label = data[:, -1, -1].to(self.device)
            weights = data[:, -1, 0].to(self.device)  # Assuming weights are at index 0 in the 4th dimension

            pred = self.model(feature.float())
            loss = self.loss_fn(pred, label, weights)  # Here is -r2, which is
            losses.append(loss.item())

        return float(np.mean(losses))

    def _init_data_loader(self, data, shuffle=True, drop_last=True):
        '''
        :param data: This is an instance of TSDataSampler, which contains the data and sampling logic.
        :param shuffle: This parameter is used to create the DailyBatchSamplerRandom sampler, indicating whether to shuffle the data at the beginning of each epoch.
        :param drop_last: This parameter indicates whether to drop the last incomplete batch.
        :return: DataLoader instance: DataLoader uses the sampler to generate data indices and loads data from TSDataSampler according to these indices. drop_last=True means that if the data cannot be divided evenly, the last incomplete batch will be dropped.
        '''
        sampler = DailyBatchSamplerRandom(data, shuffle)
        data_loader = DataLoader(data, sampler=sampler, drop_last=drop_last)
        return data_loader

    def load_param(self, param_path):
        self.model.load_state_dict(torch.load(param_path, map_location=self.device))
        self.fitted = True

    def fit(self, dl_train, dl_valid, patience=5, min_delta=0.001):
        # Initialize training and validation data loaders
        train_loader = self._init_data_loader(dl_train, shuffle=True, drop_last=True)
        valid_loader = self._init_data_loader(dl_valid, shuffle=False, drop_last=True)

        self.fitted = True  # Mark the models as trained
        best_param = None  # Used to store the best models parameters
        best_val_loss = float('inf')  # Initialize the best validation loss to infinity
        epochs_no_improve = 0  # Record the number of epochs without improvement

        for step in range(self.n_epochs):
            # Train one epoch and calculate the training loss
            train_loss = self.train_epoch(train_loader)  # Since we want to minimize the loss, return the negative R-squared value
            # Validate one epoch and calculate the validation loss
            val_loss = self.test_epoch(valid_loader)

            # Print the training and validation loss for the current epoch
            # print("Epoch %d, train_loss_ %.6f, valid_loss %.6f " % (step, train_loss, val_loss))
            print("Epoch %d, train_weighted_r_2 %.6f, valid_weighted_r_2 %.6f " % (step, -train_loss, -val_loss))

            # If the validation loss improves
            if val_loss < best_val_loss - min_delta:
                best_val_loss = val_loss  # Update the best validation loss
                best_param = copy.deepcopy(self.model.state_dict())  # Save the current best models parameters
                epochs_no_improve = 0  # Reset the number of epochs without improvement
            else:
                epochs_no_improve += 1  # Increase the number of epochs without improvement

            # If the number of epochs without improvement reaches the patience value, stop training early
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {step}")
                break
                
            # torch.save(best_param, f'{self.save_path}{self.save_prefix}master_{step}.pkl')

        # Save the models parameters 
        torch.save(best_param, f'{self.save_path}{self.save_prefix}master.pkl')

        # Load the best models parameters
        if best_param is not None:
            self.model.load_state_dict(best_param)

    def predict(self, dl_test):
        if not self.fitted:
            raise ValueError("models is not fitted yet!")
    
        test_loader = self._init_data_loader(dl_test, shuffle=False, drop_last=False)
    
        preds = []
        # labels = []
        weighted_r2s = []
    
        self.model.eval()
        for data in test_loader:
            data = torch.squeeze(data, dim=0)
            feature = data[:, :, 1:-1].to(self.device)
            label = data[:, -1, -1]
            weight = data[:, -1, 0]
            with torch.no_grad():
                pred = self.model(feature.float()).detach().cpu().numpy()
            preds.append(pred.ravel())
            weighted_r2 = calc_weighted_r2(pred, label.detach().numpy(), weight.detach().numpy())
            weighted_r2s.append(weighted_r2)

        predictions = pd.Series(np.concatenate(preds), index=dl_test.get_index())


        return predictions, label, np.mean(weighted_r2s)

## Utils for MASTER models

In [22]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model) # Create a zero tensor pe of shape (max_len, d_model) to store positional encodings.
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # Create a float tensor position from 0 to max_len-1 and expand it to shape (max_len, 1).
                                                                          # This is done for broadcasting operations later.
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term) # Assign computed sine values to even columns of pe
        pe[:, 1::2] = torch.cos(position * div_term) # Assign computed cosine values to odd columns of pe
        self.register_buffer("pe", pe) # Register pe as a persistent buffer, making it part of the module but not models parameters.

    def forward(self, x):
        return x + self.pe[:x.shape[1], :] # In the forward pass, add the input data x with the corresponding positional encoding self.pe[:x.shape[1], :].
                                          # Here, self.pe[:x.shape[1], :] extracts the positional encodings matching the time steps of the input data.

class SAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout):
        super().__init__()

        self.d_model = d_model
        self.nhead = nhead
        self.temperature = math.sqrt(self.d_model/nhead) # Used to scale the attention score matrix

        self.qtrans = nn.Linear(d_model, d_model, bias=False)
        self.ktrans = nn.Linear(d_model, d_model, bias=False)
        self.vtrans = nn.Linear(d_model, d_model, bias=False)

        attn_dropout_layer = []
        for i in range(nhead):
            attn_dropout_layer.append(nn.Dropout(p=dropout))
        self.attn_dropout = nn.ModuleList(attn_dropout_layer)

        # Input LayerNorm
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)

        # FFN layerNorm
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(d_model, d_model),
            nn.Dropout(p=dropout)
        )

    def forward(self, x):
        x = self.norm1(x)
        q = self.qtrans(x).transpose(0,1) # Transpose dimensions 0 and 1, resulting in shape (8,472,256)
        k = self.ktrans(x).transpose(0,1)
        v = self.vtrans(x).transpose(0,1)

        dim = int(self.d_model/self.nhead) # 256/2
        att_output = []
        for i in range(self.nhead):
            if i==self.nhead-1:
                qh = q[:, :, i * dim:]
                kh = k[:, :, i * dim:]
                vh = v[:, :, i * dim:]
            else:
                qh = q[:, :, i * dim:(i + 1) * dim]
                kh = k[:, :, i * dim:(i + 1) * dim]
                vh = v[:, :, i * dim:(i + 1) * dim]

            atten_ave_matrixh = torch.softmax(torch.matmul(qh, kh.transpose(1, 2)) / self.temperature, dim=-1) # Shape (8, 472, 472).
                                                                                                           # Represents attention weights for each stock j from all stocks i at every time step t.
            if self.attn_dropout:
                atten_ave_matrixh = self.attn_dropout[i](atten_ave_matrixh)
            att_output.append(torch.matmul(atten_ave_matrixh, vh).transpose(0, 1)) # Different from TA here
        att_output = torch.concat(att_output, dim=-1)

        # FFN
        xt = x + att_output
        xt = self.norm2(xt)
        att_output = xt + self.ffn(xt)

        return att_output # Shape (472,8,256)

class TAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.qtrans = nn.Linear(d_model, d_model, bias=False) # Input and output dimensions are both d_model
        self.ktrans = nn.Linear(d_model, d_model, bias=False)
        self.vtrans = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = []
        if dropout > 0: # If dropout parameter is greater than 0, create a Dropout layer for each attention head and store them in self.attn_dropout list.
            for i in range(nhead):
                self.attn_dropout.append(nn.Dropout(p=dropout))
            self.attn_dropout = nn.ModuleList(self.attn_dropout)

        # Input LayerNorm
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5) # Layer normalization, normalizing all feature dimensions for each sample
        # FFN layerNorm
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(d_model, d_model),
            nn.Dropout(p=dropout)
        )

    def forward(self, x):
        x = self.norm1(x) # Input is normalized, resulting in shape (482,8,256)
        q = self.qtrans(x)
        k = self.ktrans(x)
        v = self.vtrans(x)

        dim = int(self.d_model / self.nhead) # Dimension per attention head
        att_output = []
        for i in range(self.nhead): # Ensures that each attention head processes different feature subspaces.
            if i==self.nhead-1:
                qh = q[:, :, i * dim:]
                kh = k[:, :, i * dim:]
                vh = v[:, :, i * dim:]
            else:
                qh = q[:, :, i * dim:(i + 1) * dim] # Extract partial data from dimension d_model along the third dimension, resulting in shape (482,8,64)
                kh = k[:, :, i * dim:(i + 1) * dim]
                vh = v[:, :, i * dim:(i + 1) * dim]
            atten_ave_matrixh = torch.softmax(torch.matmul(qh, kh.transpose(1, 2)), dim=-1) # Calculate attention weight matrix, shape (482,8,8).
            # Each element atten_weight[n, t_i, t_j] represents the attention weight from time step t_i to time step t_j in the nth sample.
            if self.attn_dropout:
                atten_ave_matrixh = self.attn_dropout[i](atten_ave_matrixh)
            att_output.append(torch.matmul(atten_ave_matrixh, vh)) # Each element represents the weighted sum result at time step t_i in the nth sample, shape (482,8,64)
        att_output = torch.concat(att_output, dim=-1) # Concatenate outputs from all attention heads along the last dimension, forming the final attention output tensor, shape (482,8,256)

        # FFN
        xt = x + att_output # Residual connection adding original input x and attention output att_output, resulting in shape (482,8,256)
        xt = self.norm2(xt) # Layer normalization on xt for each feature dimension, making mean close to 0 and variance close to 1, resulting in shape (482,8,256)
        att_output = xt + self.ffn(xt) # Non-linear transformation through feed-forward network ffn and another residual connection, resulting in shape (482,8,256)

        return att_output # Final output, shape (482,8,256)

class TemporalAttention(nn.Module):
    '''
    Its main purpose is to extract the most important information from features across multiple time steps and generate a comprehensive temporal embedding.
    '''
    def __init__(self, d_model):
        super().__init__()
        self.trans = nn.Linear(d_model, d_model, bias=False)

    def forward(self, z): # z: [N, T, D], e.g.(475,8,256)
        h = self.trans(z) # [N, T, D], e.g.(475,8,256), Transform the input temporal embeddings z into new representations h
        query = h[:, -1, :].unsqueeze(-1) # [N, D]-> [N, D, 1] e.g.(475,256,1), Extract the feature vector of the last time step of each sample,
                                         # representing the latest state or trend used to measure the importance of other time step embeddings.
        lam = torch.matmul(h, query).squeeze(-1)  # [N, T, 1] --> [N, T] e.g.(475,8), Attention score matrix where each row corresponds to the attention scores of all time steps in one sample,
                                                 # representing the importance scores of each time step.
        lam = torch.softmax(lam, dim=1).unsqueeze(1) # Apply softmax on scores over time steps to get weights [N, T] --> [N, 1, T], e.g.(475,1,8)
        output = torch.matmul(lam, z).squeeze(1)  # Generate a comprehensive temporal embedding by weighted sum of all time step features according to the probability distribution.
                                                # Get final output [N, 1, T], [N, T, D] --> [N, D], e.g.(475,256)
        return output # Comprehensive temporal embedding, e.g.(475,256)





In [23]:
class MASTER(nn.Module):
    def __init__(self, d_feat=124, d_model=256, t_nhead=4, s_nhead=2, T_dropout_rate=0.5, S_dropout_rate=0.5):
        super(MASTER, self).__init__()
    
        self.layers = nn.Sequential(
            # feature layer
            nn.Linear(d_feat, d_model),
            PositionalEncoding(d_model),
            # intra-stock aggregation
            TAttention(d_model=d_model, nhead=t_nhead, dropout=T_dropout_rate),
            # inter-stock aggregation
            SAttention(d_model=d_model, nhead=s_nhead, dropout=S_dropout_rate),
            TemporalAttention(d_model=d_model),
            # decoder
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        output = self.layers(x).squeeze(-1)
        #output = output - torch.mean(output, dim=0, keepdim=True) # 对模型的输出进行均值归一化，这段代码确保了输出张量在第0维度上的均值为零。
        
        return output


In [24]:
class MASTERModel(SequenceModel):
    def __init__(
            self, d_feat: int = 20, d_model: int = 64, t_nhead: int = 4, s_nhead: int = 2,
            T_dropout_rate=0.5, S_dropout_rate=0.5, **kwargs,
    ):
        super(MASTERModel, self).__init__(**kwargs)
        self.d_model = d_model
        self.d_feat = d_feat

        self.T_dropout_rate = T_dropout_rate
        self.S_dropout_rate = S_dropout_rate
        self.t_nhead = t_nhead
        self.s_nhead = s_nhead

        self.init_model()

    def init_model(self):
        self.model = MASTER(d_feat=self.d_feat, d_model=self.d_model, t_nhead=self.t_nhead, s_nhead=self.s_nhead,
                                   T_dropout_rate=self.T_dropout_rate, S_dropout_rate=self.S_dropout_rate)
        super(MASTERModel, self).init_model()

# main

In [25]:
save_prefix = '/models' # save models path 

d_feat = 79 # 除去第0维的weights和第81维的label，共79个feature
d_model = 256
t_nhead = 4
s_nhead = 2
dropout = 0.5

n_epoch = 1
lr = 8e-6
GPU = 0
seed = 0

In [26]:
model = MASTERModel(
    d_feat=d_feat, d_model=d_model, t_nhead=t_nhead, s_nhead=s_nhead, T_dropout_rate=dropout, S_dropout_rate=dropout,
    n_epochs=n_epoch, lr=lr, GPU=GPU, seed=seed,
    save_path='../models/MASTER', save_prefix=save_prefix
)

Using device: mps


### Pretrained Model and Further Training

Before this point, I have pre-trained a model using the data from `processed_data/Processed_data_1300.parquet` and saved it at `../model/MASTER/modelmaster_pretrain.pkl`. To facilitate your work, you can load this pretrained model and proceed with further training or predictions using the same dataset or additional data.

By starting with a pretrained model, you can save considerable time and computational resources while achieving faster convergence and potentially better performance in downstream tasks.

In [27]:
# Train
param_path = '../models/MASTER/modelmaster_pretrain.pkl' # load pretrained models
model.load_param(param_path)
model.fit(train_dataset, valid_dataset) # train the models
print("Model Trained.")

Epoch 0, train_weighted_r_2 0.002278, valid_weighted_r_2 -0.017949 
Model Trained.


In [28]:
# Test
predictions, labels, weighted_R2 = model.predict(test_dataset)

In [29]:
print(weighted_R2)

0.00395758221071373


In [30]:
print(predictions)

time_id    symbol_id
1698571.0  0.0          0.050502
           1.0          0.056272
           2.0          0.038639
           3.0         -0.008448
           4.0         -0.036208
                          ...   
1698967.0  35.0        -0.008232
           36.0        -0.008785
           37.0        -0.009557
           38.0        -0.013512
           39.0         0.302611
Length: 15880, dtype: float32
